# Treino do modelo de classificação de lesões de pele (HAM10000)

TCC de Engenharia de Computação (UNILINS, 2026): *Plataforma Inteligente para Educação Dermatológica*.

Este notebook treina a rede que substitui o simulador do microsserviço de IA. Ao final, ele gera um `.zip` com:

- `modelo_ham10000.onnx`: o modelo, já com a calibração embutida (a saída são probabilidades);
- `classes.json`: a ordem das classes, os nomes em português e os parâmetros de pré-processamento;
- `metricas.json`, `metricas_por_classe.csv` e `historico_treino.csv`: os números do treino e do teste;
- figuras em PNG para o texto do TCC (distribuição das classes, curvas de treino, matriz de confusão, confiabilidade).

**Como rodar no Google Colab**

1. *Ambiente de execução > Alterar o tipo de ambiente de execução* e escolha **GPU T4**.
2. *Ambiente de execução > Executar tudo*. O treino completo leva cerca de 30 minutos.
3. Se o download do dataset pedir login, envie o `kaggle.json` quando a célula pedir (Kaggle > *Settings* > *API* > *Create New Token*).
4. No fim, o navegador baixa o `modelo_ham10000.zip`.

Para conferir se tudo funciona antes do treino longo, mude `MODO_TESTE = True` na célula de configuração. Assim ele usa poucas imagens e uma época de cada fase, em poucos minutos.

**Dataset:** HAM10000, de Tschandl, Rosendahl e Kittler (2018), *Scientific Data*, v. 5, 180161. Licença para uso não comercial.

> O modelo é uma ferramenta de apoio. A saída dele não é diagnóstico: quem decide é o médico.

## 1. Ambiente e configuração

In [ ]:
!pip install -q kagglehub onnx onnxscript onnxruntime

In [ ]:
import copy
import json
import os
import random
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# ---------------- Configuração ----------------
MODO_TESTE = False         # True = rodada rápida (poucas imagens, 1 época por fase) só para conferir
PASTA_DADOS = None         # None = baixa do Kaggle; ou o caminho de uma pasta com o HAM10000 extraído
COPIAR_PARA_DRIVE = False  # True = copia o .zip final para o Google Drive (pede autorização)
PASTA_SAIDA = Path("saida")

SEMENTE = 42
TAMANHO = 224              # lado da imagem de entrada da rede
TAMANHO_CACHE_TREINO = 256 # o treino guarda as imagens um pouco maiores para recortar trechos aleatórios
LOTE = 64
EPOCAS_FASE1 = 3           # fase 1: só a camada final
EPOCAS_FASE2 = 25          # fase 2: a rede inteira (com parada antecipada)
PACIENCIA = 6              # épocas sem melhora antes de parar a fase 2
LR_FASE1 = 1e-3
LR_FASE2 = 2e-4

if MODO_TESTE:
    EPOCAS_FASE1, EPOCAS_FASE2 = 1, 1

def fixar_semente(semente):
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    torch.cuda.manual_seed_all(semente)

fixar_semente(SEMENTE)
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USAR_AMP = DISPOSITIVO.type == "cuda"   # precisão mista: treino mais rápido na GPU
TRABALHADORES = min(4, os.cpu_count() or 2) if os.name != "nt" else 0
torch.backends.cudnn.benchmark = True
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print(f"PyTorch {torch.__version__} | dispositivo: {DISPOSITIVO}")
if DISPOSITIVO.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATENÇÃO: sem GPU. Ative a GPU T4 em 'Alterar o tipo de ambiente de execução'.")

# Estilo das figuras (salvas em PNG para o TCC; a legenda ABNT fica no Word)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e3e3e3", "grid.linewidth": 0.8,
})
COR_PRINCIPAL = "#1f77b4"   # azul
COR_SECUNDARIA = "#ff7f0e"  # laranja (par azul/laranja é seguro para daltonismo)
COR_REFERENCIA = "#8a8a8a"

## 2. Classes

A ordem desta lista é a ordem da saída do modelo. Os nomes em português são as chaves do JSON que o `POST /predict` devolve.

In [ ]:
CLASSES = [
    ("akiec", "Ceratose actínica / Doença de Bowen"),
    ("bcc",   "Carcinoma basocelular"),
    ("bkl",   "Lesão queratósica benigna"),
    ("df",    "Dermatofibroma"),
    ("mel",   "Melanoma"),
    ("nv",    "Nevo melanocítico"),
    ("vasc",  "Lesão vascular"),
]
CODIGOS = [codigo for codigo, _ in CLASSES]
NOMES = [nome for _, nome in CLASSES]
INDICE = {codigo: i for i, codigo in enumerate(CODIGOS)}
NUM_CLASSES = len(CLASSES)

## 3. Download do dataset

Usa o `kagglehub` para baixar o dataset `kmader/skin-cancer-mnist-ham10000`. Se o download sem login falhar, a célula pede o `kaggle.json`.

In [ ]:
ID_KAGGLE = "kmader/skin-cancer-mnist-ham10000"

def baixar_dataset():
    import kagglehub
    try:
        return Path(kagglehub.dataset_download(ID_KAGGLE))
    except Exception as erro:
        print(f"O download sem login falhou ({erro}).")
        print("Envie o kaggle.json (Kaggle > Settings > API > Create New Token).")
        from google.colab import files
        enviado = files.upload()
        pasta_kaggle = Path.home() / ".kaggle"
        pasta_kaggle.mkdir(exist_ok=True)
        (pasta_kaggle / "kaggle.json").write_bytes(next(iter(enviado.values())))
        os.chmod(pasta_kaggle / "kaggle.json", 0o600)
        return Path(kagglehub.dataset_download(ID_KAGGLE))

pasta = Path(PASTA_DADOS) if PASTA_DADOS else baixar_dataset()
arquivo_metadados = next(pasta.rglob("HAM10000_metadata.csv"))
caminhos = {p.stem: p for p in pasta.rglob("*.jpg")}

df = pd.read_csv(arquivo_metadados)
df["caminho"] = df["image_id"].map(caminhos)
faltando = int(df["caminho"].isna().sum())
assert faltando == 0, f"{faltando} imagens do metadados não foram encontradas"
assert df["dx"].isin(CODIGOS).all(), "o metadados tem uma classe fora da lista"
df["rotulo"] = df["dx"].map(INDICE)

if MODO_TESTE:
    df = df.sample(frac=1, random_state=SEMENTE).groupby("dx").head(60).reset_index(drop=True)

print(f"{len(df)} imagens de {df['lesion_id'].nunique()} lesões")
contagem = df["dx"].value_counts().reindex(CODIGOS)
display(pd.DataFrame({
    "classe": NOMES,
    "imagens": contagem.values,
    "%": (100 * contagem.values / len(df)).round(1),
}, index=CODIGOS))

In [ ]:
# Figura: distribuição das classes (mostra o desbalanceamento)
ordem = np.argsort(contagem.values)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh([NOMES[i] for i in ordem], contagem.values[ordem], color=COR_PRINCIPAL, height=0.6)
for posicao, valor in enumerate(contagem.values[ordem]):
    ax.text(valor, posicao, f"  {valor}", va="center", color="#333333")
ax.set_xlabel("Número de imagens")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, contagem.max() * 1.15)
fig.savefig(PASTA_SAIDA / "distribuicao_classes.png")
plt.show()

## 4. Divisão em treino, validação e teste por lesão

O HAM10000 tem várias fotos da mesma lesão. Se a divisão fosse feita por imagem, fotos da mesma lesão cairiam no treino e no teste, e o resultado do teste ficaria otimista. Por isso a divisão é feita por `lesion_id` com o `StratifiedGroupKFold`: os dados são separados em 20 partes, que mantêm a proporção das classes, e cada lesão fica inteira numa parte só. Três partes vão para o teste (15%), três para a validação (15%) e o restante para o treino (70%).

In [ ]:
divisor = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEMENTE)
parte = np.empty(len(df), dtype=int)
for i, (_, indices) in enumerate(divisor.split(df, df["rotulo"], groups=df["lesion_id"])):
    parte[indices] = i
df["conjunto"] = np.where(parte < 3, "teste", np.where(parte < 6, "validacao", "treino"))

# Conferência: nenhuma lesão aparece em mais de um conjunto
lesoes = {nome: set(df.loc[df["conjunto"] == nome, "lesion_id"]) for nome in ["treino", "validacao", "teste"]}
assert not (lesoes["treino"] & lesoes["validacao"])
assert not (lesoes["treino"] & lesoes["teste"])
assert not (lesoes["validacao"] & lesoes["teste"])
print("OK: nenhuma lesão aparece em dois conjuntos.")

tabela = pd.crosstab(df["dx"], df["conjunto"]).reindex(index=CODIGOS, columns=["treino", "validacao", "teste"])
tabela.loc["total"] = tabela.sum()
display(tabela)
df[["image_id", "lesion_id", "dx", "conjunto"]].to_csv(PASTA_SAIDA / "divisao.csv", index=False)

## 5. Pré-processamento

As duas funções abaixo são **as mesmas que o microsserviço vai usar** no Render, com Pillow e numpy:

1. corrige a orientação da foto (EXIF, comum em fotos de celular);
2. recorta o quadrado central;
3. redimensiona para 224 × 224;
4. normaliza com a média e o desvio padrão do ImageNet.

A validação e o teste passam exatamente por elas, então as métricas refletem o que o serviço vai fazer.

Para acelerar o treino, as imagens já recortadas ficam na memória. As de treino ficam com 256 px, porque o aumento de dados recorta trechos aleatórios delas.

In [ ]:
MEDIA = np.array([0.485, 0.456, 0.406], dtype=np.float32)
DESVIO = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def recortar_e_redimensionar(img, tamanho):
    """Corrige a orientação (EXIF), recorta o quadrado central e redimensiona."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    largura, altura = img.size
    lado = min(largura, altura)
    esquerda, topo = (largura - lado) // 2, (altura - lado) // 2
    img = img.crop((esquerda, topo, esquerda + lado, topo + lado))
    return img.resize((tamanho, tamanho), Image.Resampling.BILINEAR)

def normalizar(img):
    """Imagem RGB -> array float32 (3, altura, largura) normalizado."""
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = (x - MEDIA) / DESVIO
    return np.ascontiguousarray(x.transpose(2, 0, 1), dtype=np.float32)

def carregar(caminho, tamanho):
    with Image.open(caminho) as img:
        return np.asarray(recortar_e_redimensionar(img, tamanho))

def carregar_conjunto(nome, tamanho):
    linhas = df[df["conjunto"] == nome]
    with ThreadPoolExecutor(max_workers=8) as executor:
        imagens = list(executor.map(lambda c: carregar(c, tamanho), linhas["caminho"]))
    return np.stack(imagens), linhas["rotulo"].to_numpy(dtype=np.int64)

inicio = time.time()
X_treino, y_treino = carregar_conjunto("treino", TAMANHO_CACHE_TREINO)
X_val, y_val = carregar_conjunto("validacao", TAMANHO)
X_teste, y_teste = carregar_conjunto("teste", TAMANHO)
print(f"Imagens carregadas em {time.time() - inicio:.0f} s: "
      f"treino {len(X_treino)}, validação {len(X_val)}, teste {len(X_teste)}")

In [ ]:
# Figura: um exemplo de cada classe (após o recorte)
fig, eixos = plt.subplots(1, NUM_CLASSES, figsize=(14, 2.6))
for k, ax in enumerate(eixos):
    ax.imshow(X_treino[np.flatnonzero(y_treino == k)[0]])
    ax.set_title(NOMES[k].replace(" / ", " /\n"), fontsize=8)
    ax.axis("off")
fig.savefig(PASTA_SAIDA / "exemplos_classes.png")
plt.show()

## 6. Aumento de dados e carregadores

No treino, cada imagem passa por transformações aleatórias a cada época: recorte de um trecho (60% a 100% da área), espelhamento, rotação de 90°, 180° ou 270° e variação de brilho, contraste, saturação e tom. A rede vê variações das mesmas lesões, o que reduz o sobreajuste. A variação de cor também ajuda um pouco com a diferença entre a foto do dermatoscópio e a do celular.

In [ ]:
class Rotacao90:
    """Gira a imagem em 0°, 90°, 180° ou 270° (lesões não têm 'lado certo')."""
    ROTACOES = [None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270]

    def __call__(self, img):
        rotacao = self.ROTACOES[int(torch.randint(0, 4, (1,)))]
        return img if rotacao is None else img.transpose(rotacao)

transformacao_treino = transforms.Compose([
    transforms.RandomResizedCrop(TAMANHO, scale=(0.6, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    Rotacao90(),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA.tolist(), DESVIO.tolist()),
])

class ImagensHAM(Dataset):
    def __init__(self, imagens, rotulos, transformacao=None):
        self.imagens, self.rotulos, self.transformacao = imagens, rotulos, transformacao

    def __len__(self):
        return len(self.rotulos)

    def __getitem__(self, i):
        img = Image.fromarray(self.imagens[i])
        if self.transformacao is not None:
            x = self.transformacao(img)
        else:
            x = torch.from_numpy(normalizar(img))  # validação/teste: igual ao serviço
        return x, self.rotulos[i]

opcoes = dict(batch_size=LOTE, num_workers=TRABALHADORES, pin_memory=USAR_AMP,
              persistent_workers=TRABALHADORES > 0)
carregador_treino = DataLoader(ImagensHAM(X_treino, y_treino, transformacao_treino),
                               shuffle=True, drop_last=True, **opcoes)
carregador_val = DataLoader(ImagensHAM(X_val, y_val), shuffle=False, **opcoes)
carregador_teste = DataLoader(ImagensHAM(X_teste, y_teste), shuffle=False, **opcoes)

## 7. Modelo e treino

**Transferência de aprendizado:** a EfficientNet-B0 já vem treinada no ImageNet (1,2 milhão de fotos) e sabe reconhecer bordas, texturas e formas. Só a última camada é trocada por uma de 7 saídas.

O treino tem duas fases:

1. **Fase 1:** só a camada nova é treinada (3 épocas), para ela não estragar os pesos prontos no começo.
2. **Fase 2:** a rede inteira é ajustada com taxa de aprendizado menor, que cai em formato de cosseno. A fase para se a validação passar 6 épocas sem melhorar.

**Desbalanceamento:** a perda é ponderada pelo inverso da frequência de cada classe. Errar um dermatofibroma (raro) pesa muito mais que errar um nevo (cerca de 2/3 do dataset). Sem isso, a rede aprenderia a dizer "nevo" para quase tudo.

**Critério de escolha:** fica o modelo da época com maior **acurácia balanceada** na validação, que é a média da sensibilidade das 7 classes. A acurácia comum seria enganosa: um modelo que só responde "nevo" já acerta cerca de 67%.

In [ ]:
def criar_modelo():
    modelo = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    modelo.classifier[1] = nn.Linear(modelo.classifier[1].in_features, NUM_CLASSES)
    return modelo.to(DISPOSITIVO)

modelo = criar_modelo()

contagem_treino = np.bincount(y_treino, minlength=NUM_CLASSES)
pesos_classes = len(y_treino) / (NUM_CLASSES * contagem_treino)
PESOS = torch.tensor(pesos_classes, dtype=torch.float32)
criterio = nn.CrossEntropyLoss(weight=PESOS.to(DISPOSITIVO))
display(pd.DataFrame({"classe": NOMES, "imagens no treino": contagem_treino,
                      "peso na perda": pesos_classes.round(2)}, index=CODIGOS))

In [ ]:
escalador = torch.amp.GradScaler("cuda", enabled=USAR_AMP)

def calcular_logits(carregador):
    """Roda o modelo num conjunto e devolve as saídas brutas (logits) e os rótulos."""
    modelo.eval()
    saidas, rotulos = [], []
    with torch.no_grad():
        for x, y in carregador:
            with torch.autocast(DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_AMP):
                saidas.append(modelo(x.to(DISPOSITIVO, non_blocking=True)).float().cpu())
            rotulos.append(y)
    return torch.cat(saidas), torch.cat(rotulos)

def treinar_uma_epoca(otimizador):
    modelo.train()
    soma, total = 0.0, 0
    for x, y in carregador_treino:
        x, y = x.to(DISPOSITIVO, non_blocking=True), y.to(DISPOSITIVO, non_blocking=True)
        otimizador.zero_grad(set_to_none=True)
        with torch.autocast(DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_AMP):
            perda = criterio(modelo(x), y)
        escalador.scale(perda).backward()
        escalador.step(otimizador)
        escalador.update()
        soma += perda.item() * len(y)
        total += len(y)
    return soma / total

historico = []
melhor = {"acuracia_balanceada": -1.0, "epoca": None, "estado": None}

def executar_fase(fase, epocas, taxa, so_camada_final, paciencia=None):
    for p in modelo.features.parameters():
        p.requires_grad = not so_camada_final
    parametros = [p for p in modelo.parameters() if p.requires_grad]
    otimizador = torch.optim.AdamW(parametros, lr=taxa, weight_decay=1e-2)
    agendador = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=epocas)
    sem_melhora = 0
    for _ in range(epocas):
        inicio = time.time()
        perda_treino = treinar_uma_epoca(otimizador)
        agendador.step()
        logits, rotulos = calcular_logits(carregador_val)
        perda_val = F.cross_entropy(logits, rotulos, weight=PESOS).item()
        acuracia_bal = balanced_accuracy_score(rotulos, logits.argmax(1))
        epoca = len(historico) + 1
        historico.append({"epoca": epoca, "fase": fase, "perda_treino": perda_treino,
                          "perda_validacao": perda_val, "acuracia_balanceada_validacao": acuracia_bal,
                          "segundos": time.time() - inicio})
        marca = ""
        if acuracia_bal > melhor["acuracia_balanceada"]:
            melhor.update(acuracia_balanceada=acuracia_bal, epoca=epoca,
                          estado=copy.deepcopy(modelo.state_dict()))
            torch.save(melhor["estado"], PASTA_SAIDA / "melhor_modelo.pt")
            sem_melhora, marca = 0, "  <- melhor até agora"
        else:
            sem_melhora += 1
        print(f"época {epoca:2d} (fase {fase}) | perda treino {perda_treino:.3f} | "
              f"perda val {perda_val:.3f} | acurácia balanceada val {acuracia_bal:.3f} | "
              f"{time.time() - inicio:.0f} s{marca}")
        if paciencia and sem_melhora >= paciencia:
            print(f"Parada antecipada: {paciencia} épocas sem melhora.")
            break

inicio_treino = time.time()
executar_fase(1, EPOCAS_FASE1, LR_FASE1, so_camada_final=True)
executar_fase(2, EPOCAS_FASE2, LR_FASE2, so_camada_final=False, paciencia=PACIENCIA)
print(f"\nTreino concluído em {(time.time() - inicio_treino) / 60:.1f} min. "
      f"Melhor época: {melhor['epoca']} (acurácia balanceada na validação {melhor['acuracia_balanceada']:.3f}).")

modelo.load_state_dict(melhor["estado"])
pd.DataFrame(historico).to_csv(PASTA_SAIDA / "historico_treino.csv", index=False)

In [ ]:
# Figura: curvas de treino
h = pd.DataFrame(historico)
inicio_fase2 = h.loc[h["fase"] == 2, "epoca"].min()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(h["epoca"], h["perda_treino"], color=COR_PRINCIPAL, lw=2, marker="o", ms=4, label="Treino")
ax1.plot(h["epoca"], h["perda_validacao"], color=COR_SECUNDARIA, lw=2, marker="o", ms=4, label="Validação")
ax1.set_xlabel("Época")
ax1.set_ylabel("Perda (entropia cruzada ponderada)")
ax1.legend(frameon=False)
ax2.plot(h["epoca"], h["acuracia_balanceada_validacao"], color=COR_PRINCIPAL, lw=2, marker="o", ms=4)
ax2.plot(melhor["epoca"], melhor["acuracia_balanceada"], "o", ms=10, mfc="none", mec="#222222", mew=1.5)
ax2.annotate(f"melhor: {melhor['acuracia_balanceada']:.3f}", (melhor["epoca"], melhor["acuracia_balanceada"]),
             textcoords="offset points", xytext=(0, 10), ha="center", color="#222222")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia balanceada (validação)")
for ax in (ax1, ax2):
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.axvline(inicio_fase2 - 0.5, color=COR_REFERENCIA, ls=":", lw=1)
    ax.text(inicio_fase2 - 0.4, 0.98, "fase 2", transform=ax.get_xaxis_transform(),
            va="top", color=COR_REFERENCIA, fontsize=8)
fig.savefig(PASTA_SAIDA / "curvas_treino.png")
plt.show()

## 8. Calibração das probabilidades (*temperature scaling*)

Redes neurais costumam ser confiantes demais: dizem 95% quando acertam bem menos que isso. O *temperature scaling* (Guo et al., 2017) divide as saídas brutas por um número *T* ajustado na validação. Isso não muda a classe escolhida, só deixa as porcentagens mais honestas. O *T* fica embutido no ONNX.

Duas métricas medem o efeito:

- **NLL** (log-verossimilhança negativa): quanto menor, melhor;
- **ECE** (erro de calibração esperado): a diferença média entre a confiança e a taxa de acerto.

Observação para o texto: como a perda foi ponderada, o modelo tende a dar um pouco mais de probabilidade às classes raras (melanoma incluso) do que a frequência delas no dataset justificaria. Num sistema de apoio isso é aceitável, porque é preferível um alerta a mais do que um melanoma ignorado.

In [ ]:
logits_val, rot_val = calcular_logits(carregador_val)
logits_teste, rot_teste = calcular_logits(carregador_teste)

def ajustar_temperatura(logits, rotulos):
    """Testa valores de T entre 0,1 e 10 e fica com o de menor NLL na validação."""
    candidatos = torch.cat([torch.logspace(-1, 1, 401), torch.tensor([1.0])])
    nll = torch.stack([F.cross_entropy(logits / t, rotulos) for t in candidatos])
    return float(candidatos[nll.argmin()])

def faixas_confiabilidade(probs, rotulos, n_faixas=10):
    confianca, previsto = probs.max(1)
    acerto = (previsto == rotulos).float()
    limites = torch.linspace(0, 1, n_faixas + 1)
    faixas = []
    for a, b in zip(limites[:-1], limites[1:]):
        m = (confianca > a) & (confianca <= b)
        if m.any():
            faixas.append((confianca[m].mean().item(), acerto[m].mean().item(), m.float().mean().item()))
    return faixas

def ece(probs, rotulos):
    return sum(peso * abs(acc - conf) for conf, acc, peso in faixas_confiabilidade(probs, rotulos, 15))

T = ajustar_temperatura(logits_val, rot_val)
print(f"Temperatura ajustada: T = {T:.3f}")

calibracao = {}
for nome, logits, rotulos in [("validacao", logits_val, rot_val), ("teste", logits_teste, rot_teste)]:
    for rotulo_t, t in [("antes", 1.0), ("depois", T)]:
        probs = F.softmax(logits / t, dim=1)
        calibracao[f"nll_{nome}_{rotulo_t}"] = F.cross_entropy(logits / t, rotulos).item()
        calibracao[f"ece_{nome}_{rotulo_t}"] = ece(probs, rotulos)
display(pd.DataFrame({
    "NLL antes": [calibracao["nll_validacao_antes"], calibracao["nll_teste_antes"]],
    "NLL depois": [calibracao["nll_validacao_depois"], calibracao["nll_teste_depois"]],
    "ECE antes": [calibracao["ece_validacao_antes"], calibracao["ece_teste_antes"]],
    "ECE depois": [calibracao["ece_validacao_depois"], calibracao["ece_teste_depois"]],
}, index=["validação", "teste"]).round(4))

In [ ]:
# Figura: diagrama de confiabilidade no teste
fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot([0, 1], [0, 1], color=COR_REFERENCIA, ls="--", lw=1, label="Calibração perfeita")
for t, cor, rotulo in [(1.0, COR_SECUNDARIA, "Antes da calibração"), (T, COR_PRINCIPAL, f"Depois (T = {T:.2f})")]:
    faixas = faixas_confiabilidade(F.softmax(logits_teste / t, dim=1), rot_teste)
    ax.plot([f[0] for f in faixas], [f[1] for f in faixas], color=cor, lw=2, marker="o", ms=5, label=rotulo)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Confiança média da classe prevista")
ax.set_ylabel("Taxa de acerto")
ax.legend(frameon=False, loc="upper left")
fig.savefig(PASTA_SAIDA / "confiabilidade.png")
plt.show()

## 9. Avaliação no conjunto de teste

O teste não foi usado em nenhuma decisão até aqui (nem na escolha da época, nem na calibração), então mede como o modelo se sai com lesões que nunca viu.

- **Sensibilidade** (*recall*): das lesões de uma classe, quantas o modelo acertou. É a métrica mais importante para o melanoma.
- **Especificidade:** das lesões que não são daquela classe, quantas o modelo não confundiu com ela.
- **Precisão:** quando o modelo diz "melanoma", quantas vezes é mesmo melanoma.
- **AUC:** área sob a curva ROC (1 = separação perfeita, 0,5 = acaso).

In [ ]:
probs_teste = F.softmax(logits_teste / T, dim=1).numpy()
y_real = rot_teste.numpy()
y_prev = probs_teste.argmax(1)
rotulos_idx = list(range(NUM_CLASSES))

matriz = confusion_matrix(y_real, y_prev, labels=rotulos_idx)
precisao, sensibilidade, f1, suporte = precision_recall_fscore_support(
    y_real, y_prev, labels=rotulos_idx, zero_division=0)
especificidade, auc = [], []
for k in rotulos_idx:
    verdadeiro_negativo = matriz.sum() - matriz[k, :].sum() - matriz[:, k].sum() + matriz[k, k]
    falso_positivo = matriz[:, k].sum() - matriz[k, k]
    especificidade.append(verdadeiro_negativo / (verdadeiro_negativo + falso_positivo))
    positivos = (y_real == k)
    auc.append(roc_auc_score(positivos, probs_teste[:, k]) if 0 < positivos.sum() < len(positivos) else float("nan"))

por_classe = pd.DataFrame({
    "classe": NOMES, "imagens": suporte, "sensibilidade": sensibilidade,
    "especificidade": especificidade, "precisao": precisao, "f1": f1, "auc": auc,
}, index=CODIGOS)
por_classe.to_csv(PASTA_SAIDA / "metricas_por_classe.csv")

resumo = {
    "acuracia": accuracy_score(y_real, y_prev),
    "acuracia_balanceada": balanced_accuracy_score(y_real, y_prev),
    "auc_macro": float(np.nanmean(auc)),
}
print(f"Acurácia:            {resumo['acuracia']:.3f}")
print(f"Acurácia balanceada: {resumo['acuracia_balanceada']:.3f}")
print(f"AUC média:           {resumo['auc_macro']:.3f}")
mel = INDICE["mel"]
print(f"Melanoma: sensibilidade {sensibilidade[mel]:.3f} | especificidade {especificidade[mel]:.3f} | AUC {auc[mel]:.3f}")
display(por_classe.round(3))

In [ ]:
# Figura: matriz de confusão (cada linha soma 100%)
matriz_norm = matriz / matriz.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(8.5, 7))
imagem = ax.imshow(matriz_norm, cmap="Blues", vmin=0, vmax=1)
for i in rotulos_idx:
    for j in rotulos_idx:
        cor = "white" if matriz_norm[i, j] > 0.55 else "#222222"
        ax.text(j, i, f"{100 * matriz_norm[i, j]:.0f}%\n({matriz[i, j]})", ha="center", va="center",
                fontsize=8, color=cor)
ax.set_xticks(rotulos_idx, NOMES, rotation=40, ha="right")
ax.set_yticks(rotulos_idx, NOMES)
ax.set_xlabel("Classe prevista pelo modelo")
ax.set_ylabel("Classe real (diagnóstico do dataset)")
ax.grid(False)
for lado in ax.spines.values():
    lado.set_visible(False)
barra = fig.colorbar(imagem, ax=ax, fraction=0.046, pad=0.04)
barra.set_label("Proporção das lesões da classe real")
fig.savefig(PASTA_SAIDA / "matriz_confusao.png")
plt.show()

In [ ]:
# Tudo em um JSON, para o TCC e para o README
def contar(nome):
    linhas = df[df["conjunto"] == nome]
    return {"imagens": int(len(linhas)), "lesoes": int(linhas["lesion_id"].nunique())}

metricas = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "modo_teste": MODO_TESTE,
    "modelo": "EfficientNet-B0 pré-treinada no ImageNet, ajuste fino em duas fases",
    "dataset": "HAM10000 (Tschandl, Rosendahl e Kittler, 2018)",
    "divisao": {"criterio": "lesion_id (StratifiedGroupKFold, 70/15/15)",
                "treino": contar("treino"), "validacao": contar("validacao"), "teste": contar("teste")},
    "treino": {"epocas_executadas": len(historico), "melhor_epoca": melhor["epoca"],
               "acuracia_balanceada_validacao": melhor["acuracia_balanceada"],
               "lote": LOTE, "taxa_fase1": LR_FASE1, "taxa_fase2": LR_FASE2,
               "pesos_classes": dict(zip(CODIGOS, pesos_classes.round(4).tolist()))},
    "calibracao": {"temperatura": T, **calibracao},
    "teste": {**resumo,
              "por_classe": {c: {k: (None if pd.isna(v) else (v if isinstance(v, str) else float(v)))
                                 for k, v in por_classe.loc[c].items()} for c in CODIGOS},
              "matriz_confusao": {"ordem": CODIGOS, "valores": matriz.tolist()}},
    "versoes": {"torch": torch.__version__, "torchvision": __import__("torchvision").__version__},
}
with open(PASTA_SAIDA / "metricas.json", "w", encoding="utf-8") as arquivo:
    json.dump(metricas, arquivo, ensure_ascii=False, indent=2)
print("metricas.json salvo.")

## 10. Exportação para ONNX e teste igual ao do serviço

O modelo é exportado com a temperatura e a *softmax* embutidas, então a saída do ONNX já são as 7 probabilidades. No Render, o serviço só precisa de `onnxruntime`, Pillow e numpy, sem PyTorch, e cabe nos 512 MB do plano gratuito.

Depois da exportação, o notebook carrega o `.onnx` com o `onnxruntime`, processa imagens de teste com as mesmas funções do serviço e compara o resultado com o do PyTorch. Se o pré-processamento ou a exportação tiverem algum problema, ele aparece aqui, e não no Render.

In [ ]:
import onnx
import onnxruntime as ort

class ModeloComProbabilidades(nn.Module):
    """Rede + temperatura + softmax: a saída já é o vetor de probabilidades."""
    def __init__(self, rede, temperatura):
        super().__init__()
        self.rede = rede
        self.register_buffer("temperatura", torch.tensor(temperatura, dtype=torch.float32))

    def forward(self, x):
        return torch.softmax(self.rede(x) / self.temperatura, dim=1)

modelo_export = ModeloComProbabilidades(copy.deepcopy(modelo).float().cpu(), T).eval()
exemplo = torch.zeros(1, 3, TAMANHO, TAMANHO)
caminho_onnx = PASTA_SAIDA / "modelo_ham10000.onnx"
argumentos = dict(input_names=["entrada"], output_names=["probabilidades"], opset_version=17,
                  dynamic_axes={"entrada": {0: "lote"}, "probabilidades": {0: "lote"}})
try:
    torch.onnx.export(modelo_export, (exemplo,), str(caminho_onnx), dynamo=False, **argumentos)
except Exception as erro:  # versões futuras do PyTorch podem não ter mais o exportador clássico
    print(f"Exportador clássico indisponível ({erro}); usando o novo.")
    torch.onnx.export(modelo_export, (exemplo,), str(caminho_onnx), dynamo=True, **argumentos)

# Garante um arquivo único (sem pesos em arquivo .data separado) e válido
modelo_onnx = onnx.load(str(caminho_onnx))
onnx.checker.check_model(modelo_onnx)
onnx.save(modelo_onnx, str(caminho_onnx), save_as_external_data=False)
for extra in PASTA_SAIDA.glob("modelo_ham10000.onnx.*"):
    extra.unlink()
print(f"{caminho_onnx.name}: {caminho_onnx.stat().st_size / 1e6:.1f} MB")

configuracao = {
    "modelo": caminho_onnx.name,
    "entrada": {"nome": "entrada", "formato": "NCHW float32", "tamanho": TAMANHO,
                "recorte": "quadrado central", "redimensionamento": "bilinear",
                "escala": "pixel / 255", "media": [0.485, 0.456, 0.406], "desvio": [0.229, 0.224, 0.225]},
    "saida": {"nome": "probabilidades", "temperatura_embutida": round(T, 4)},
    "classes": [{"codigo": codigo, "nome": nome} for codigo, nome in CLASSES],
}
with open(PASTA_SAIDA / "classes.json", "w", encoding="utf-8") as arquivo:
    json.dump(configuracao, arquivo, ensure_ascii=False, indent=2)

In [ ]:
sessao = ort.InferenceSession(str(caminho_onnx), providers=["CPUExecutionProvider"])

def prever_como_servico(caminho_imagem):
    """O que o POST /predict vai fazer: arquivo -> pré-processamento -> ONNX -> JSON."""
    with Image.open(caminho_imagem) as img:
        x = normalizar(recortar_e_redimensionar(img, TAMANHO))[None]
    probabilidades = sessao.run(None, {"entrada": x})[0][0]
    return {nome: round(float(p), 4) for nome, p in zip(NOMES, probabilidades)}, x

amostra = df[df["conjunto"] == "teste"].head(30)
diferencas, concordancia = [], []
for posicao, caminho_imagem in enumerate(amostra["caminho"]):
    resposta, x = prever_como_servico(caminho_imagem)
    p_onnx = np.array(list(resposta.values()))
    with torch.no_grad():
        p_torch = modelo_export(torch.from_numpy(x))[0].numpy()
    diferencas.append(np.abs(sessao.run(None, {"entrada": x})[0][0] - p_torch).max())
    concordancia.append(p_onnx.argmax() == probs_teste[posicao].argmax())

print(f"Maior diferença ONNX x PyTorch: {max(diferencas):.2e}")
print(f"Mesma classe que na avaliação do teste: {np.mean(concordancia):.0%} de {len(amostra)} imagens")
assert max(diferencas) < 1e-3, "o ONNX não bate com o PyTorch"

resposta, _ = prever_como_servico(amostra["caminho"].iloc[0])
print(f"\nExemplo de resposta do /predict (classe real: {NOMES[amostra['rotulo'].iloc[0]]}):")
print(json.dumps(resposta, ensure_ascii=False, indent=2))
print(f"Soma: {sum(resposta.values()):.4f}")

## 11. Download dos resultados

Gera o `modelo_ham10000.zip` e baixa pelo navegador. O `melhor_modelo.pt` (pesos do PyTorch) não entra no zip porque o serviço não precisa dele. Ele fica na pasta `saida/` do Colab enquanto a sessão estiver aberta.

In [ ]:
caminho_zip = Path("modelo_ham10000.zip")
with zipfile.ZipFile(caminho_zip, "w", zipfile.ZIP_DEFLATED) as zip_saida:
    for arquivo in sorted(PASTA_SAIDA.iterdir()):
        if arquivo.suffix != ".pt":
            zip_saida.write(arquivo, arquivo.name)
            print(f"  {arquivo.name}")
print(f"{caminho_zip}: {caminho_zip.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import drive, files
except ImportError:
    print(f"Fora do Colab: o arquivo está em {caminho_zip.resolve()}")
else:
    if COPIAR_PARA_DRIVE:
        import shutil
        drive.mount("/content/drive")
        shutil.copy(caminho_zip, "/content/drive/MyDrive/modelo_ham10000.zip")
        print("Cópia salva em MyDrive/modelo_ham10000.zip")
    files.download(str(caminho_zip))

## Próximos passos

1. Traga o `modelo_ham10000.zip` para o projeto. O `.onnx` e o `classes.json` vão para o `ai-service`, e o `main.py` passa a usar o modelo real no lugar do simulador, sem mudar o contrato do `POST /predict`.
2. Use as figuras e o `metricas_por_classe.csv` no capítulo de resultados do TCC.
3. Declare a limitação: o HAM10000 tem imagens de dermatoscópio, e o sistema recebe fotos de celular. O desempenho com fotos comuns tende a ser menor que o medido aqui. O sistema não é um dispositivo médico.